# Day 2, Notebook 1: package the logic, survive bad data

Yesterday your answers lived in cells. Today they become things you can call again, on records you have not seen yet.

Run this notebook top to bottom. Every failure in it is deliberate and every one is captured, so the notebook keeps running and you can read the error text at your own pace. Read each error before you read the fix.

Position in the day:

`[inline cell] > [packaged decision] > [read the failure] > [log the rejection] > [cross the boundary]`

The map below is where this notebook sits in the day and what it adds. Every
notebook in the programme opens on the same pair, so you know where you are before you read a line.

The cell that draws it also brings in the programme's helper. `scripts/c2kit.py` is found by
walking up from this notebook's own folder, which is what lets the same file run whether you
pressed Run All here or a script ran it for you. The helper loads the day's data from `../data/`,
draws every diagram you see in these notebooks, and runs the checks that tell you a cell did what
it claimed.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["functions and errors", "files and formats", "hands-on: trace the calls", "hands-on: the truncated feed"], lit=0, title="the day's notebooks", show=False),
    kit.flow(["the cell you wrote three times", "the rule gets a name", "return against print", "catch what you expected", "the rejection is a deliverable"], title="what this notebook adds", show=False),
)

## Setup

One cell, at the top, so this notebook runs cold in a fresh Codespace.

`csv` is today's topic. `traceback` is plumbing that lets a failure print itself without stopping the notebook, and you are not expected to learn it today.

In [2]:
import csv
import traceback

DATA_DIR = "../data"
ORDERS_CSV = f"{DATA_DIR}/C2_W01_D02_orders_STUDENT.csv"

with open(ORDERS_CSV) as f:
    orders = list(csv.DictReader(f))

def show_failure(broken):
    """Run a function that is meant to fail and print its real traceback."""
    try:
        broken()
    except Exception:
        print(traceback.format_exc())

print(f"{len(orders)} records loaded")
print(orders[0])

30 records loaded
{'order_id': 'KR4200', 'customer_id': 'C1645', 'segment': 'Retail-Core', 'amount': '4500', 'status': 'returned', 'order_date': '2026-08-03', 'discount': ''}


In [3]:
kit.check("thirty rows came out of the CSV", len(orders) == 30, f"got {len(orders)}")
kit.check("every value read from a CSV arrives as text",
          all(isinstance(r["amount"], str) for r in orders))
kit.check("two amounts will not convert, which is today's work",
          len([r for r in orders if not r["amount"].strip().lstrip("-").isdigit()]) == 2,
          "KR4210 spells its amount and KR4214 has none")

## Where this is going, before we build any of it

Here is the thing you will have written by the end of the next hour. Do not read it closely. Watch what it does.

In [4]:
def clean_record(record):
    """Return one record with its amount as a number, or raise ValueError saying what arrived."""
    keeper = dict(record)
    keeper["amount"] = int(record["amount"])
    return keeper


three_defective = [orders[10], orders[14], orders[0]]

for record in three_defective:
    try:
        print("kept    ", clean_record(record)["order_id"], clean_record(record)["amount"])
    except ValueError as e:
        print("rejected", record["order_id"], "because:", e)

rejected KR4210 because: invalid literal for int() with base 10: 'twelve'
rejected KR4214 because: invalid literal for int() with base 10: ''
kept     KR4200 4500


One function, three records, three different outcomes, and the third one tells you why. That is today's deliverable.

Now we build it from where you were yesterday.

In [5]:
kit.flow(["the cell you wrote three times", "the rule gets a name", "return against print", "catch what you expected", "the rejection is a deliverable"], lit=0)

## Section 1: the cell you have already written three times

This is Monday's code, character for character. It worked yesterday.

Today it is pointed at today's file, and today's file came from somewhere you do not control.

In [6]:
def monday_cell():
    total = 0
    for r in orders:
        if r["status"] == "delivered":
            total = total + int(r["amount"])
    print(total)

show_failure(monday_cell)

Traceback (most recent call last):
  File "/tmp/ipykernel_5160/812909849.py", line 13, in show_failure
    broken()
  File "/tmp/ipykernel_5160/3770030529.py", line 5, in monday_cell
    total = total + int(r["amount"])
                    ^^^^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: 'twelve'



The code did not change. The data did. That is most of this job.

We come back to the error itself in section 4. For now, notice the shape of the problem before the error: the rule lives in a cell. To use it on a second dataset you copy the cell. To change the rule you edit every copy.

A rule that lives in three cells is three rules, and the copy you forget is the one that ships.

In [7]:
kit.flow(["the cell you wrote three times", "the rule gets a name", "return against print", "catch what you expected", "the rejection is a deliverable"], lit=1)

## Section 2: the same rule with a name

Section 1 plus one new element: the rule gets a name and one home.

Mental model:

```
records  ->  [ delivered_total ]  ->  a number
             the promise: give me records, I give you a number
```

The function can see only what you hand it. That is all of scope you need today.

In [8]:
def delivered_total(some_orders):
    total = 0
    for r in some_orders:
        if r["status"] == "delivered":
            total = total + int(r["amount"])
    return total


def call_on_everything():
    return delivered_total(orders)

# It fails the same way the cell did, for the same reason.
show_failure(call_on_everything)

Traceback (most recent call last):
  File "/tmp/ipykernel_5160/812909849.py", line 13, in show_failure
    broken()
  File "/tmp/ipykernel_5160/1127745375.py", line 10, in call_on_everything
    return delivered_total(orders)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_5160/1127745375.py", line 5, in delivered_total
    total = total + int(r["amount"])
                    ^^^^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: 'twelve'



The failure moved with the rule, which is the point. Naming code does not fix code. It gives the fix one place to live.

Work on the eight records that do convert, so the shape of the function is visible before we handle the defects.

In [9]:
first_eight = orders[:8]
print(delivered_total(first_eight))

6095


In [10]:
kit.check("the same rule now has one home and one name", callable(delivered_total))
kit.check("it survives being handed a smaller file", delivered_total(first_eight) > 0,
          f"first eight give Rs {delivered_total(first_eight)}")

### The compact form of the same loop

You will meet this notation constantly in other people's code, so meet it once here.

A list comprehension builds a list in one line. It is the loop you already know, written differently, and it is notation rather than a new idea.

In [11]:
# The loop you know
amounts = []
for r in first_eight:
    amounts.append(int(r["amount"]))

# The same thing, one line
amounts_again = [int(r["amount"]) for r in first_eight]

print(amounts)
print(amounts_again)
print("same result:", amounts == amounts_again)

[4500, 2395, 1360, 1440, 2905, 2095, 920, 1420]
[4500, 2395, 1360, 1440, 2905, 2095, 920, 1420]
same result: True


Use whichever reads more clearly to the person maintaining it. That is the whole rule. We go no deeper into comprehensions today.

### Milestone: where this shows up in production

Every data pipeline you will ever read is a chain of named functions like `delivered_total`, one per decision, called in order. The name is what lets a colleague review your logic without reading it. When an incident report says "the rule changed in one place and three callers were not updated", that report is about functions.

### Interview question this milestone just made answerable

"What does a function actually buy you over a copied block of code?"

Your answer has three parts now: one home for the rule, a name you can say out loud, and reuse on inputs you have not seen.

In [12]:
kit.flow(["the cell you wrote three times", "the rule gets a name", "return against print", "catch what you expected", "the rejection is a deliverable"], lit=2)

## Section 3: return against print

Section 2 plus one new element: what the function hands back.

`print` shows a human. `return` hands a value to the next line of code. Confusing the two is the most common first-week bug in any language.

In [13]:
def fix_printing(record):
    print(record["order_id"])

def fix_returning(record):
    return record["order_id"]

printed = fix_printing(orders[0])
returned = fix_returning(orders[0])

print("printed holds:", repr(printed))
print("returned holds:", repr(returned))

KR4200
printed holds: None
returned holds: 'KR4200'


### The deliberate failure

`fix_printing` showed you something and handed back nothing. Nothing is `None`. Now use it as though it were a value.

In [14]:
def use_the_nothing():
    return fix_printing(orders[0])["id"]

show_failure(use_the_nothing)

KR4200
Traceback (most recent call last):
  File "/tmp/ipykernel_5160/812909849.py", line 13, in show_failure
    broken()
  File "/tmp/ipykernel_5160/1696257024.py", line 2, in use_the_nothing
    return fix_printing(orders[0])["id"]
           ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
TypeError: 'NoneType' object is not subscriptable



Read the last line first:

```
TypeError: 'NoneType' object is not subscriptable
```

The type is `TypeError`. The value it was holding was `None`. `None` came from a function that printed instead of returning.

### The fix

In [15]:
record_id = fix_returning(orders[0])
print("the caller can use this:", record_id)
print("and pass it straight on:", record_id.startswith("10"))

the caller can use this: KR4200
and pass it straight on: False


In [16]:
kit.check("the printing version hands back nothing at all",
          fix_printing(orders[0]) is None)
kit.check("the returning version hands back something the caller can use",
          isinstance(record_id, str) and record_id != "")

KR4200


### The two shapes, side by side

In [17]:
kit.side_by_side(
    kit.sequence(["the caller", "fix_printing", "your screen"],
                 [("the caller", "fix_printing", "one record"),
                  ("fix_printing", "your screen", "prints the id"),
                  ("fix_printing", "the caller", "None")],
                 title="print shows you, and returns nothing", show=False),
    kit.sequence(["the caller", "fix_returning", "the next call"],
                 [("the caller", "fix_returning", "one record"),
                  ("fix_returning", "the caller", "the id, as a value"),
                  ("the caller", "the next call", "passes it straight on")],
                 title="return hands the answer back", show=False),
)

Rule for the rest of the programme: if the caller needs the answer, the function returns it. Print is for you, standing at the screen.

In [18]:
kit.flow(["the cell you wrote three times", "the rule gets a name", "return against print", "catch what you expected", "the rejection is a deliverable"], lit=3)

## Section 4: catch the failure you expected

Section 3 plus one new element: surviving a bad value on purpose.

A traceback is the interpreter telling you where it stopped and what it was holding. Read it bottom-up:

```
last line       what went wrong         the exception type and the value
line above      where                   the line that is yours
everything else how you got there       the call chain
```

In [19]:
def convert_a_word():
    return int("twelve")

show_failure(convert_a_word)

Traceback (most recent call last):
  File "/tmp/ipykernel_5160/812909849.py", line 13, in show_failure
    broken()
  File "/tmp/ipykernel_5160/1519425734.py", line 2, in convert_a_word
    return int("twelve")
           ^^^^^^^^^^^^^
ValueError: invalid literal for int() with base 10: 'twelve'



Three questions, every time:

1. What is the exception type? `ValueError`.
2. Which line is mine? The one calling `int`.
3. What value was it holding? `'twelve'`.

The third question is the one people skip, and it is the one that names the record.

### Catching it narrowly

In [20]:
def normalise_amount(raw):
    """Convert an amount, or raise ValueError with the interpreter's own wording."""
    return int(raw)

converted = 0
failed = 0
for r in orders:
    try:
        normalise_amount(r["amount"])
        converted += 1
    except ValueError:
        failed += 1

print(f"converted {converted}, failed {failed}")

converted 28, failed 2


### Raising on purpose, with your own message

`int` raised for you there. Sometimes the value converts and is still unacceptable, and then you raise yourself, with a message that says what your rule was.

In [21]:
def normalise_amount_strict(raw):
    """Convert an amount and refuse anything below zero."""
    value = int(raw)
    if value < 0:
        raise ValueError(f"amount below zero: {value}")
    return value


def try_a_negative():
    return normalise_amount_strict("-4500")

print(normalise_amount_strict("4500"))
show_failure(try_a_negative)

4500
Traceback (most recent call last):
  File "/tmp/ipykernel_5160/812909849.py", line 13, in show_failure
    broken()
  File "/tmp/ipykernel_5160/3689895507.py", line 10, in try_a_negative
    return normalise_amount_strict("-4500")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_5160/3689895507.py", line 5, in normalise_amount_strict
    raise ValueError(f"amount below zero: {value}")
ValueError: amount below zero: -4500



Notice the wording. `amount below zero: -4500` names your rule and the value that broke it, which is what the person reading your rejects file needs. You will want this one for tonight's take-home.

### The deliberate failure that does not look like one

This next cell runs. It prints a number. The number is wrong and nothing on screen says so.

In [22]:
total = 0
for r in orders:
    try:
        total += int(r["amount"])
    except:
        pass
print(f"Processed {len(orders)} records. Total: {total}")

Processed 30 records. Total: 53745


Now the honest version of exactly the same work.

In [23]:
total = 0
kept = 0
for r in orders:
    try:
        total += int(r["amount"])
    except ValueError:
        continue
    kept += 1
print(f"Clean {kept}, rejected {len(orders) - kept}, total {total}")

Clean 28, rejected 2, total 53745


In [24]:
kit.check("both cells reported the same total", total == 53745, f"Rs {total}")
kit.check("only the honest one reports how many records it read", kept == 28, f"{kept} kept")
kit.check("the two records it could not read are the two planted defects",
          len(orders) - kept == 2)

### What the bare except costs, drawn

In [25]:
kit.tree(
    {"label": "int(r['amount']) raises on this record",
     "branches": [
         ("except: pass", {"label": "Rs 53,745 reported as though 30 records went in"}),
         ("except ValueError", {"label": "Rs 53,745 reported, and 28 of 30 stated beside it"}),
         ("no handler at all", {"label": "the loop stops and you know immediately"})]},
    taken=["except ValueError"], title="the same total, three different claims")

Both cells printed `53745`. The first one claimed thirty records went into it. Two did not.

That is the whole argument. A crash costs you an hour of your own time. A plausible wrong number costs you a quarter, because nobody goes looking for a number that looks fine.

The bare `except` also swallows the failures you did not think of: a typo in a key name, an interrupted keyboard, a bug three functions down.

### Milestone: where this shows up in production

Knight Capital, 1 August 2012, lost about USD 440 million in 45 minutes. A deployment reused an old flag and the system did not fail loudly. It kept trading, at speed, on the wrong rule.

Public Health England, October 2020, dropped 15,841 COVID cases from reporting when a CSV was converted into an old Excel format with a hard row limit. Nothing crashed. The rows past the limit were silently discarded.

### Interview question this milestone just made answerable

"Why is a bare `except` worse than letting the code crash?"

Answer with the two lines above as evidence: both printed the same total, and only one of them told the truth about how many records it covered.

In [26]:
kit.flow(["the cell you wrote three times", "the rule gets a name", "return against print", "catch what you expected", "the rejection is a deliverable"], lit=4)

## Section 5: the rejection is a deliverable

Section 4 plus one new element: the bad record goes somewhere, with a reason.

You have three choices when a record will not convert. Fix it silently, drop it silently, or set it aside with a reason. Only the third survives a question from someone who was not in the room.

This is where `clean_record` from the top of the notebook gets built for real, and then called over a whole list.

In [27]:
def clean_record(record):
    """Return one record with its amount as a number, or raise ValueError saying what arrived."""
    keeper = dict(record)
    keeper["amount"] = normalise_amount(record["amount"])
    return keeper


def clean_records(rows):
    """Call clean_record on every row and keep the failures, each with its reason."""
    clean = []
    rejects = []
    for r in rows:
        try:
            clean.append(clean_record(r))
        except ValueError as e:
            rejects.append({"order_id": r["order_id"], "reason": str(e)})
    return clean, rejects


clean, rejects = clean_records(orders)
print(f"input {len(orders)}, clean {len(clean)}, rejected {len(rejects)}")
for row in rejects:
    print(row)

input 30, clean 28, rejected 2
{'order_id': 'KR4210', 'reason': "invalid literal for int() with base 10: 'twelve'"}
{'order_id': 'KR4214', 'reason': "invalid literal for int() with base 10: ''"}


`clean_record` handles one record and decides nothing about what a failure means. `clean_records` handles the list and owns that decision. A function that does both is two functions wearing one name.

`dict(record)` copies before changing. Yesterday's `b = a` trap is the reason: without the copy you would be editing the row still sitting in `rows`.

`str(e)` carries the interpreter's own wording, so nobody has to invent error messages and everyone's log reads the same.

### The reconciliation

This is the check that catches the silent loss, and you will use it every day from tomorrow.

In [28]:
assert len(clean) + len(rejects) == len(orders), "records went missing"
print(f"{len(orders)} in = {len(clean)} clean + {len(rejects)} rejected")

30 in = 28 clean + 2 rejected


In [29]:
kit.check("nothing went missing across the pass", len(clean) + len(rejects) == len(orders),
          f"{len(orders)} in, {len(clean)} clean, {len(rejects)} rejected")
kit.check("every rejection carries a reason a person can read",
          all(r.get("reason") for r in rejects))
kit.check("every cleaned amount is a number now",
          all(isinstance(r["amount"], int) for r in clean))

When that sum does not hold, something disappeared and you do not yet know what. The assertion is cheap. The missing rows are not.

### Milestone: interview question

"Your cleaning run reported zero rejects on a file you know is dirty. What do you check?"

Three checks, in order: is the rejects list actually being appended to, is the `except` narrow enough to be reached, and does input equal clean plus rejected.

### What this notebook established

In [30]:
kit.table(
    ["The idea", "What proved it here"],
    [["A function is a decision you can call again", "delivered_total ran on thirty records and on eight"],
     ["print shows you, return hands back", "fix_printing(orders[0]) is None"],
     ["A bare except turns a failure into a plausible wrong claim", "Rs 53,745 from 28 records, reported as 30"],
     ["A rejection is a deliverable", "input equals clean plus rejected, asserted"]],
    caption="Day 2, notebook 1",
)
kit.flow(["the cell you wrote three times", "the rule gets a name", "return against print", "catch what you expected", "the rejection is a deliverable"], lit=4, title="the notebook, end to end")
kit.check_summary()

The idea,What proved it here
A function is a decision you can call again,delivered_total ran on thirty records and on eight
"print shows you, return hands back",fix_printing(orders[0]) is None
A bare except turns a failure into a plausible wrong claim,"Rs 53,745 from 28 records, reported as 30"
A rejection is a deliverable,"input equals clean plus rejected, asserted"


## Crux

A function is a decision you can call again. A named exception is a failure you chose to survive. A rejects log is the difference between a number and a number you can defend.

## What tomorrow does with this

`clean_record` and `clean_records` get called tomorrow on a bigger and dirtier dataset, without one edit, and the question becomes how many usable records that dataset actually has.

Carry these two functions forward. You will be asked for them by name.